### Import libraries
Spark functions for column operations; DeltaTable for merging into existing tables.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Load utilities
Brings in shared schema names, S3 path and the table name helper.

In [0]:
%run ../../01_setup/utilities

### Utilities
Shared settings (catalog, schemas, S3 path) and a helper for full table names. Loaded by every pipeline notebook with `%run`.

### Notebook parameters, paths and table names
Staging tables hold only the current batch of new orders, so silver and gold process just those.

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"{s3_base_path}/{data_source}"
landing_path = f"{base_path}/landing"
processed_path = f"{base_path}/processed"

bronze_table = table_name(bronze_schema, data_source)
silver_table = table_name(silver_schema, data_source)
gold_table = table_name(gold_schema, f"sb_fact_{data_source}")
parent_fact_table = table_name(gold_schema, "fact_orders")

bronze_staging = table_name(bronze_schema, f"staging_{data_source}")
silver_staging = table_name(silver_schema, f"staging_{data_source}")

print("Landing path:  ", landing_path)
print("Processed path:", processed_path)
print("Staging tables:", bronze_staging, silver_staging)

Landing path:   s3://sportsbar-data-lake-vkt/orders/landing
Processed path: s3://sportsbar-data-lake-vkt/orders/processed
Staging tables: fmcg.bronze.staging_orders fmcg.silver.staging_orders


## Bronze

### Read new files from landing
Stops the notebook cleanly if there are no new files.

In [0]:
landing_files = [f for f in dbutils.fs.ls(landing_path) if f.name.endswith(".csv")]
print("Files in landing:", len(landing_files))

if not landing_files:
    dbutils.notebook.exit("No new files in landing – nothing to load.")

df = (
    spark.read.options(header=True, inferSchema=True)
    .csv(f"{landing_path}/*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("New rows:", df.count())
display(df.limit(5))

Files in landing: 16
New rows: 4885


order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FDEC818102602,"Tuesday, December 16, 2025",789102,25891502,196.0,2026-11-05T05:16:06.556Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",789102,25891503,null,2026-11-05T05:16:06.556Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",789102,25891602,147.0,2026-11-05T05:16:06.556Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",789102,25891101,337.0,2026-11-05T05:16:06.556Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",INVALID,25891202,202.0,2026-11-05T05:16:06.556Z,orders_2025_12_16.csv,23060


### Append to bronze and save the batch to bronze staging
Bronze keeps the full history; staging holds only this batch.

In [0]:
(
    df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("append")
    .saveAsTable(bronze_table)
)

(
    df.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(bronze_staging)
)

print("Rows in bronze:        ", spark.table(bronze_table).count())
print("Rows in bronze staging:", spark.table(bronze_staging).count())

Rows in bronze:         61777
Rows in bronze staging: 4885


### Move loaded files from landing to processed

In [0]:
for f in landing_files:
    dbutils.fs.mv(f.path, f"{processed_path}/{f.name}")

print("Files moved to processed:", len(landing_files))

Files moved to processed: 16


## Silver

### Load the new batch from bronze staging

In [0]:
df_orders = spark.table(bronze_staging)
print("Rows in this batch:", df_orders.count())

Rows in this batch: 4885


### Clean the batch
Same rules as the full load: remove missing quantities, fix customer IDs, standardise dates, remove duplicates.

In [0]:
# 1. Remove orders without a quantity
print("Rows without quantity:", df_orders.filter(F.col("order_qty").isNull()).count())
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

# 2. Clean customer_id
customer_text = F.col("customer_id").cast("string")
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(customer_text.rlike("^[0-9]+$"), customer_text).otherwise(F.lit("999999"))
)
print("Orders with invalid customer_id:", df_orders.filter(F.col("customer_id") == "999999").count())

# 3. Standardise the order date
date_text = F.regexp_replace(F.col("order_placement_date").cast("string"), r"^[A-Za-z]+,\s*", "")
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date(date_text, "yyyy/MM/dd"),
        F.try_to_date(date_text, "dd-MM-yyyy"),
        F.try_to_date(date_text, "dd/MM/yyyy"),
        F.try_to_date(date_text, "yyyy-MM-dd"),
        F.try_to_date(date_text, "MMMM dd, yyyy"),
    )
)
print("Unreadable dates:", df_orders.filter(F.col("order_placement_date").isNull()).count())

# 4. Remove duplicates and convert product_id to text
rows_before = df_orders.count()
df_orders = df_orders.dropDuplicates(
    ["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"]
)
print("Duplicates removed:", rows_before - df_orders.count())

df_orders = df_orders.withColumn("product_id", F.col("product_id").cast("string"))

df_orders.agg(
    F.min("order_placement_date").alias("first_date"),
    F.max("order_placement_date").alias("last_date"),
    F.countDistinct("order_placement_date").alias("days_with_orders"),
).show()

Rows without quantity: 233
Orders with invalid customer_id: 500
Unreadable dates: 0
Duplicates removed: 337
+----------+----------+----------------+
|first_date| last_date|days_with_orders|
+----------+----------+----------------+
|2025-12-16|2025-12-31|              16|
+----------+----------+----------------+



### Attach product_code
Orders without a matching product are shown, then dropped.

In [0]:
df_products = spark.table(table_name(silver_schema, "products")).select("product_id", "product_code")

df_unmatched = df_orders.join(df_products, "product_id", "left_anti")
print("Orders without a matching product:", df_unmatched.count())
display(df_unmatched.groupBy("product_id").count().orderBy("product_id"))

df_joined = df_orders.join(df_products, "product_id", "inner")
print("Rows after join:", df_joined.count())

Orders without a matching product: 441


product_id,count
25891502,211
77777777,80
88888888,77
99999999,73


Rows after join: 3874


### Merge into silver and save the batch to silver staging
Silver keeps all cleaned orders; staging holds only this batch for the gold step.

In [0]:
(
    DeltaTable.forName(spark, silver_table).alias("target")
    .merge(
        df_joined.alias("source"),
        """target.order_placement_date = source.order_placement_date
           AND target.order_id = source.order_id
           AND target.product_code = source.product_code
           AND target.customer_id = source.customer_id""",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

(
    df_joined.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(silver_staging)
)

print("Rows in silver:        ", spark.table(silver_table).count())
print("Rows in silver staging:", spark.table(silver_staging).count())

Rows in silver:         48645
Rows in silver staging: 3874


## Gold

### Merge the batch into sb_fact_orders

In [0]:
df_gold = spark.table(silver_staging).select(
    "order_id",
    F.col("order_placement_date").alias("date"),
    F.col("customer_id").alias("customer_code"),
    "product_code",
    "product_id",
    F.col("order_qty").alias("sold_quantity"),
)

(
    DeltaTable.forName(spark, gold_table).alias("target")
    .merge(
        df_gold.alias("source"),
        """target.date = source.date
           AND target.order_id = source.order_id
           AND target.product_code = source.product_code
           AND target.customer_code = source.customer_code""",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Rows in gold:", spark.table(gold_table).count())

Rows in gold: 48645


## Merge into parent

### Find the months touched by this batch
Only these months need recalculating in Atlikon's table.

In [0]:
batch_months = (
    spark.table(silver_staging)
    .select(F.trunc("order_placement_date", "MM").alias("date"))
    .distinct()
)

display(batch_months)

date
2025-12-01


### Recalculate monthly totals for those months
Uses ALL days of each affected month from `sb_fact_orders`, not just the new batch, so the month total is complete.

In [0]:
target_types = dict(spark.table(parent_fact_table).dtypes)

df_monthly = (
    spark.table(gold_table)
    .withColumn("date", F.trunc("date", "MM"))
    .join(batch_months, "date", "inner")
    .groupBy("date", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumn("sold_quantity", F.col("sold_quantity").cast(target_types["sold_quantity"]))
    .withColumn("customer_code", F.col("customer_code").cast(target_types["customer_code"]))
)

print("Monthly rows to merge:", df_monthly.count())
display(df_monthly.limit(5))

Monthly rows to merge: 612


date,product_code,customer_code,sold_quantity
2025-12-01,0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,789202,3179
2025-12-01,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,999999,18053
2025-12-01,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,789422,6006
2025-12-01,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789201,561
2025-12-01,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,789402,5496


### Merge into Atlikon's fact_orders
Existing month rows are replaced with the recalculated totals; new ones are added.

In [0]:
rows_before = spark.table(parent_fact_table).count()

(
    DeltaTable.forName(spark, parent_fact_table).alias("target")
    .merge(
        df_monthly.alias("source"),
        """target.date = source.date
           AND target.product_code = source.product_code
           AND target.customer_code = source.customer_code""",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

rows_after = spark.table(parent_fact_table).count()
print("Rows before:", rows_before)
print("Rows after: ", rows_after)
print("Rows added: ", rows_after - rows_before)

Rows before: 96727
Rows after:  96727
Rows added:  0


In [0]:
dec_total = (
    spark.table(parent_fact_table)
    .filter(F.col("date") == "2025-12-01")
    .agg(F.sum("sold_quantity"))
    .first()[0]
)
print("December units (after batch 1):", dec_total)

December units (after batch 1): 1827212


## Cleanup
Removes the staging tables; the next run creates them fresh.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {bronze_staging}")
spark.sql(f"DROP TABLE IF EXISTS {silver_staging}")
print("Staging tables removed")

Staging tables removed
